# Import Library

In [1]:
# Referensi medis:
# [1] ACC/AHA Guidelines — Cardiovascular Risk (2019)
# [2] ADA Standards of Medical Care in Diabetes (2023)
# [3] KDIGO Clinical Practice Guideline for CKD (2022)
# [4] WHO — Haemoglobin concentrations for anaemia (2011)
# [5] Harrison's Principles of Internal Medicine, 21st Ed.
# [6] Data asli: merge_DocLab_Clean.csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load Data


In [2]:
np.random.seed(42)

df_real = pd.read_csv('merge_DocLab_Clean.csv')

if 'NO_REG' in df_real.columns:
    df_real = df_real.drop(columns=['NO_REG'])

# Hapus Jika NM_UNIT = 3 (paru-paru, tidak dipakai)
# df_real = df_real[df_real['NM_UNIT'] != 3.0].copy()

# Konversi NM_UNIT ke integer — WAJIB agar cocok dengan key PARAMS
df_real['NM_UNIT'] = df_real['NM_UNIT'].astype(int)

# Reset index — WAJIB agar assign nilai tidak gagal/misalign
df_real = df_real.reset_index(drop=True)

# Konversi UMUR_TAHUN ke numeric
df_real['UMUR_TAHUN'] = pd.to_numeric(df_real['UMUR_TAHUN'], errors='coerce')

LAB_COLS = ['cholesterol total', 'creatinin', 'fbs', 'rbs',
            'hgb', 'lymfosit%', 'mch', 'mchc', 'mcv', 'ureum', 'wbc']

# Data Cleaning

## PARAMETER DISTRIBUSI PER KELAS

In [3]:
# Format: (mean, std, min_clip, max_clip, desimal)
PARAMS = {
    # NM_UNIT = 0 : NORMAL [Ref 6]
    0: {
        'cholesterol total': (188,   35,   111,  300,  2),
        'creatinin':         (1.05,  0.30, 0.40, 2.50, 2),
        'fbs':               (105,   22,   70,   200,  1),
        'rbs':               (130,   40,   70,   300,  1),
        'hgb':               (12.5,  1.50, 7.0,  18.0, 1),
        'lymfosit%':         (27.0,  8.0,  10,   55,   1),
        'mch':               (28.2,  2.50, 18,   35,   1),
        'mchc':              (33.2,  1.50, 28,   37,   1),
        'mcv':               (85.0,  7.0,  65,   105,  1),
        'ureum':             (35.0,  15.0, 10,   100,  1),
        'wbc':               (8.5,   2.50, 3.5,  18,   1),
    },
    # NM_UNIT = 1 : JANTUNG [Ref 1,2,4,5]
    1: {
        'cholesterol total': (215,   38,   130,  340,  2),
        'creatinin':         (1.35,  0.50, 0.50, 5.00, 2),
        'fbs':               (148,   55,   72,   450,  1),
        'rbs':               (195,   85,   72,   600,  1),
        'hgb':               (11.8,  1.80, 7.0,  17.0, 1),
        'lymfosit%':         (22.0,  9.0,  5,    50,   1),
        'mch':               (28.0,  3.00, 18,   36,   1),
        'mchc':              (33.0,  2.00, 27,   37,   1),
        'mcv':               (84.0,  8.00, 60,   105,  1),
        'ureum':             (42.0,  18.0, 10,   150,  1),
        'wbc':               (9.5,   3.00, 3.5,  20,   1),
    },
    # NM_UNIT = 2 : PENYAKIT DALAM [Ref 2,3,4,6]
    2: {
        'cholesterol total': (200,   38,   120,  310,  2),
        'creatinin':         (3.00,  2.50, 0.40, 25.0, 2),
        'fbs':               (188,   80,   72,   500,  1),
        'rbs':               (220,  110,   50,   700,  1),
        'hgb':               (10.5,  1.80, 5.5,  16.0, 1),
        'lymfosit%':         (25.0,  9.0,  5,    55,   1),
        'mch':               (27.5,  3.00, 18,   36,   1),
        'mchc':              (32.0,  2.00, 24,   37,   1),
        'mcv':               (86.0,  7.00, 65,   105,  1),
        'ureum':             (78.0,  45.0, 10,   400,  1),
        'wbc':               (9.2,   3.50, 3.0,  30,   1),
    },

    # NM_UNIT = 3 : PARU-PARU
    3: {
      'cholesterol total': (180,  35,  110, 280,  2),
      'creatinin':         (1.00, 0.40, 0.40, 3.50, 2),
      'fbs':               (115,  35,   72,  300,  1),
      'rbs':               (145,  55,   70,  400,  1),
      'hgb':               (10.8, 1.80, 6.0,  16.0, 1),
      'lymfosit%':         (19.0, 7.00, 5,    40,   1),
      'mch':               (26.5, 3.00, 17,   34,   1),
      'mchc':              (31.5, 2.00, 24,   36,   1),
      'mcv':               (82.0, 8.00, 60,  100,   1),
      'ureum':             (35.0, 15.0, 10,  120,   1),
      'wbc':               (10.5, 3.50, 3.5,  25,   1),
    }
}

## IMPUTASI UMUR


In [4]:
def impute_age(df):
    df_out = df.copy().reset_index(drop=True)

    # Hitung global fallback dari semua kelas yang punya data umur
    global_age = df_out['UMUR_TAHUN'].dropna()
    fallback_mu = global_age.mean()
    fallback_sd = global_age.std() if global_age.std() > 0 else 10.0

    for unit in sorted(df_out['NM_UNIT'].unique()):
        mask_unit = df_out['NM_UNIT'] == unit
        mask      = mask_unit & df_out['UMUR_TAHUN'].isnull()
        n_missing = int(mask.sum())

        if n_missing == 0:
            print(f"  NM_UNIT={unit}: tidak ada missing umur")
            continue

        real_age = df_out.loc[mask_unit & ~df_out['UMUR_TAHUN'].isnull(), 'UMUR_TAHUN'].dropna()

        if len(real_age) >= 5:
            mu = real_age.mean()
            sd = real_age.std() if real_age.std() > 0 else 10.0
            src = "data asli kelas"
        else:
            # Kelas tidak punya data umur → pakai distribusi global
            mu, sd = fallback_mu, fallback_sd
            src = "fallback global"

        print(f"  NM_UNIT={unit}: mengisi {n_missing} missing | mu={mu:.1f}, sd={sd:.1f} [{src}]")
        filled = np.clip(np.random.normal(mu, sd, n_missing), 4, 95).round(0)
        df_out.loc[mask, 'UMUR_TAHUN'] = filled

    return df_out

## Imputasi Parameter Fitur


In [5]:
def impute_synthetic(df, params_dict, lab_cols):
    df_out = df.copy().reset_index(drop=True)

    for unit, params in params_dict.items():
        mask_unit = df_out['NM_UNIT'] == unit

        for col in lab_cols:
            if col not in params:
                continue

            mu, sd, lo, hi, dec = params[col]
            mask      = mask_unit & df_out[col].isnull()
            n_missing = int(mask.sum())

            if n_missing == 0:
                continue

            real_vals = df_out.loc[mask_unit & ~df_out[col].isnull(), col]
            if len(real_vals) >= 5:
                mu_use = real_vals.mean()
                sd_use = real_vals.std()
            else:
                mu_use = mu
                sd_use = sd

            filled = np.clip(
                np.random.normal(mu_use, sd_use * 0.8, n_missing),
                lo, hi
            ).round(dec)
            df_out.loc[mask, col] = filled

    return df_out

# Run & Save

In [6]:
print("MENJALANKAN IMPUTASI...")

print("\n[Step 1] Imputasi UMUR_TAHUN:")
df_imputed = impute_age(df_real)
print(f"  ✅ Missing umur : {df_real['UMUR_TAHUN'].isnull().sum()} → {df_imputed['UMUR_TAHUN'].isnull().sum()}")
print(f"  Range           : {df_imputed['UMUR_TAHUN'].min():.0f} - {df_imputed['UMUR_TAHUN'].max():.0f}")

print("\n[Step 2] Imputasi parameter lab:")
df_imputed = impute_synthetic(df_imputed, PARAMS, LAB_COLS)
print(f"  ✅ Missing lab tersisa : {df_imputed[LAB_COLS].isnull().sum().sum()}")

print(f"\n📊 Shape final  : {df_imputed.shape}")
print(f"NM_UNIT dist   : {df_imputed['NM_UNIT'].value_counts().sort_index().to_dict()}")
print(f"Total missing  : {df_imputed.isnull().sum().sum()}")

MENJALANKAN IMPUTASI...

[Step 1] Imputasi UMUR_TAHUN:
  NM_UNIT=0: mengisi 2173 missing | mu=57.5, sd=14.2 [fallback global]
  NM_UNIT=1: tidak ada missing umur
  NM_UNIT=2: tidak ada missing umur
  NM_UNIT=3: tidak ada missing umur
  ✅ Missing umur : 2173 → 0
  Range           : 4 - 95

[Step 2] Imputasi parameter lab:
  ✅ Missing lab tersisa : 0

📊 Shape final  : (6430, 14)
NM_UNIT dist   : {0: 2173, 1: 914, 2: 2698, 3: 645}
Total missing  : 0


In [7]:
# Save File

OUTPUT_PATH = 'merge_DocLab_hybrid.csv'
df_imputed.to_csv(OUTPUT_PATH, index=False)
print(f"\n💾 Disimpan: {OUTPUT_PATH}")


💾 Disimpan: merge_DocLab_hybrid.csv
